<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/Transformers_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Context in Attention (x vs. y vectors)**

Context in Attention: x vs. y Vectors

The Problem: 'x' Vectors (No Context)
Basic word inputs (x) are like rigid dictionary definitions. The computer's version of the word "bank" is always exactly the same, no matter what sentence it is in. It doesn't know about its neighboring words.

The Solution: 'y' Vectors (Context-Aware)
After going through the "attention" mechanism, we get 'y' vectors. A 'y' vector is a smart mixture of the original word AND the words surrounding it.

Why 'y' is Better (The "Bank" Example):

  - In the sentence "I sat by the river bank", the 'y' version of "bank" will mix with the word "river".
  - In the sentence "I put money in my bank account", the 'y' version will mix with "money" and "account".

Conclusion:
Because 'y' mixes with the whole sentence, its final form changes depending on the context. This helps the AI understand multiple meanings of the same word, just like humans do!

In [ ]:
import numpy as np

np.random.seed(42)

sentence = ["Bank", "of", "the", "river"]
seq_len = len(sentence)
d_model = 8

print(f"sentence: {sentence}")
print(f"seq_len = {seq_len}, d_model = {d_model}")

sentence: ['Bank', 'of', 'the', 'river']
seq_len = 4, d_model = 8


In [ ]:
def build_embeddings(vocab_words, d_model, seed=0):
    rng = np.random.default_rng(seed)
    vocab = sorted(set(vocab_words))
    table = {w: rng.normal(size=d_model) for w in vocab}
    return table

vocab_words = ["bank", "of", "the", "river", "account", "money", "has", "today"]
embedding_table = build_embeddings(vocab_words, d_model, seed=0)

X = np.stack([embedding_table[w.lower()] for w in sentence])
print("X shape (seq_len, d_model):", X.shape)

for w, vec in zip(sentence, X):
    print(f"  x_{w:<6} shape={vec.shape}  dims={np.round(vec, 2)}")

X shape (seq_len, d_model): (4, 8)
  x_Bank   shape=(8,)  dims=[-0.7  -1.27 -0.62  0.04 -2.33 -0.22 -1.25 -0.73]
  x_of     shape=(8,)  dims=[-0.16  0.54  0.21  0.36 -0.65 -0.13  0.78  1.49]
  x_the    shape=(8,)  dims=[ 1.8   1.32  0.36 -1.21 -0.    0.66 -1.29  0.4 ]
  x_river  shape=(8,)  dims=[-1.26  1.51  1.35  0.78  0.26 -0.31  1.46  1.96]


**Q, K, V Projections (Attention Mechanism)**

Q, K, V Projections in Attention

The Goal:
Give each word vector three specific roles to figure out how words relate to each other.

The Roles:

  - Query (Q): What is this word looking for in the sentence?
  - Key (K): What information does this word have that others might want?
  - Value (V): The actual meaning of the word that will be used.

How it Works (The Projection):

  - Take the input word vector (x_i).
  - Multiply it by 3 different learned weight matrices (W_q, W_k, W_v).
  - Note: "Learned" just means the AI figures out the best numbers for these matrices during training.

The Matrix Shortcut:
Instead of doing this one word at a time, we stack all words into one big matrix X. Then, we can find all Qs, Ks, and Vs for the whole sentence at once using simple matrix multiplicatio

In [ ]:
X

array([[-0.70373524, -1.26542147, -0.62327446,  0.04132598, -2.32503077,
        -0.21879166, -1.24591095, -0.73226735],
       [-0.15922501,  0.54084558,  0.21465912,  0.35537271, -0.65382861,
        -0.12961363,  0.78397547,  1.49343115],
       [ 1.80163487,  1.31510376,  0.35738041, -1.20831863, -0.00445413,
         0.65647494, -1.28836146,  0.39512206],
       [-1.25906553,  1.51392377,  1.34587542,  0.7813114 ,  0.26445563,
        -0.31392281,  1.45802068,  1.96025832]])

In [ ]:
d_k = d_model

rng = np.random.default_rng(1)
W_q = rng.normal(scale=0.5, size=(d_model, d_k))
W_k = rng.normal(scale=0.5, size=(d_model, d_k))
W_v = rng.normal(scale=0.5, size=(d_model, d_k))

Q = X @ W_q
K = X @ W_k
V = X @ W_v

print("X:", X.shape, " W_q:", W_q.shape, " -> Q:", Q.shape)
print("K:", K.shape, "  V:", V.shape)

X: (4, 8)  W_q: (8, 8)  -> Q: (4, 8)
K: (4, 8)   V: (4, 8)


### **Attention Weights ($w_{ij}$) Explained Simply**

* **Raw Scores (`scores[i, j]`):** Every word looks at every other word in the sentence and asks, *"How relevant are you to my meaning?"* It assigns a raw number (score) to represent that connection.
* **Scaling (`/ sqrt(d_k)`):** We divide the raw scores by a specific number to shrink them down. If the raw numbers get too large, the AI's mathematical engine gets stuck. Scaling keeps the math stable (or "well-behaved").
* **Softmax:** A mathematical trick that converts the clunky raw scores into clean percentages (ranging from 0.0 to 1.0).
* **Summing to 1:** Just like a pie chart, a single word only has 100% of its attention to give. If a sentence has 4 words, the attention percentages it hands out to those 4 words must perfectly add up to 1.0 (100%).